In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
df_sentiment = pd.DataFrame()
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/Sentiment.pkl', compression=None)

In [ ]:
df_sentiment

In [ ]:
df_sentiment.rename(columns={"Sentiment_orig":"SentimentOrig",
                "GI_Sentiment_orig":"GI_SentimentOrig"}
                ,inplace=True)

In [ ]:
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
df_sentiment = df_sentiment[df_sentiment["NewsCount"]>0]
df_sentiment['Sentiment'] = df_sentiment['Sentiment'].apply(np.float)
df_sentiment['SentimentOrig'] = df_sentiment['SentimentOrig'].apply(np.float)
df_sentiment['pos'] = df_sentiment['pos'].apply(np.float)
df_sentiment['neg'] = df_sentiment['neg'].apply(np.float)
df_sentiment['neu'] = df_sentiment['neu'].apply(np.float)
df_sentiment['compound'] = df_sentiment['compound'].apply(np.float)
df_sentiment["polarity"] = df_sentiment["polarity"].apply(np.float64)
df_sentiment["subjectivity"] = df_sentiment["subjectivity"].apply(np.float64)
df_sentiment["GI_Sentiment"] = df_sentiment["GI_Sentiment"].apply(np.float64)
df_sentiment["GI_SentimentOrig"] = df_sentiment["GI_SentimentOrig"].apply(np.float64)
df_sentiment["GI_Positive"] = df_sentiment["GI_Positive"].apply(np.float64)
df_sentiment["GI_Negative"] = df_sentiment["GI_Negative"].apply(np.float64)
df_sentiment["GI_Polarity"] = df_sentiment["GI_Polarity"].apply(np.float64)
df_sentiment["GI_Subjectivity"] = df_sentiment["GI_Subjectivity"].apply(np.float64)

df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
df_sentiment["SentimentOrig(t-1)"] = df_sentiment["SentimentOrig"].shift(periods=1)
df_sentiment["SentimentOrig(t-2)"] = df_sentiment["SentimentOrig"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["GI_Sentiment(t-1)"] = df_sentiment["GI_Sentiment"].shift(periods=1)
df_sentiment["GI_Sentiment(t-2)"] = df_sentiment["GI_Sentiment"].shift(periods=2)
df_sentiment["GI_SentimentOrig(t-1)"] = df_sentiment["GI_SentimentOrig"].shift(periods=1)
df_sentiment["GI_SentimentOrig(t-2)"] = df_sentiment["GI_SentimentOrig"].shift(periods=2)
df_sentiment["GI_Positive(t-1)"] = df_sentiment["GI_Positive"].shift(periods=1)
df_sentiment["GI_Positive(t-2)"] = df_sentiment["GI_Positive"].shift(periods=2)
df_sentiment["GI_Negative(t-1)"] = df_sentiment["GI_Negative"].shift(periods=1)
df_sentiment["GI_Negative(t-2)"] = df_sentiment["GI_Negative"].shift(periods=2)
df_sentiment["NewsCount(t-1)"] = df_sentiment["NewsCount"].shift(periods=1)
df_sentiment["NewsCount(t-2)"] = df_sentiment["NewsCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment = df_sentiment["2011-01-04":]


df_sentiment["Sentiment_ln_return"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
df_sentiment["Sentiment_ln_return(t-1)"]=df_sentiment["Sentiment_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])
df_sentiment["SentimentOrig_ln_return"]=np.log(df_sentiment["SentimentOrig"]/df_sentiment["SentimentOrig(t-1)"])
df_sentiment["SentimentOrig_ln_return(t-1)"]=df_sentiment["SentimentOrig_ln_return"].shift(periods=1)#np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()

df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos"]/df_sentiment["pos(t-1)"])
df_sentiment["pos_ln_return(t-1)"]=df_sentiment["pos_ln_return"].shift(periods=1)#np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg"]/df_sentiment["neg(t-1)"])
df_sentiment["neg_ln_return(t-1)"]=df_sentiment["neg_ln_return"].shift(periods=1)#np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu"]/df_sentiment["neu(t-1)"])
df_sentiment["neu_ln_return(t-1)"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound"]/df_sentiment["compound(t-1)"])
df_sentiment["compound_ln_return(t-1)"]=df_sentiment["compound_ln_return"].shift(periods=1)#np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

df_sentiment["polarity(t-1)"] = df_sentiment["polarity"].shift(periods=1)
df_sentiment["subjectivity(t-1)"] = df_sentiment["subjectivity"].shift(periods=1)

df_sentiment["NewsCount_ewm10"] = df_sentiment["NewsCount"].ewm(10).mean()
df_sentiment["NewsCount_ln_return"]=np.log(df_sentiment["NewsCount"]/df_sentiment["NewsCount(t-1)"])
df_sentiment["NewsCount_ln_return(t-1)"]=df_sentiment["NewsCount_ln_return"].shift(periods=1)#np.log(df_sentiment["NewsCount(t-1)"]/df_sentiment["NewsCount(t-2)"])

df_sentiment['WordCount'] = df_sentiment['lead_paragraph'].apply(lambda x:len(str(x).split())) #Number Of words in main text
df_sentiment["WordCount_ln_return"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])
df_sentiment["WordCount_ln_return(t-1)"]=df_sentiment["WordCount_ln_return"].shift(periods=1)#np.log(df_sentiment["WordCount(t-1)"]/df_sentiment["WordCount(t-2)"])

# df["GI_Sentiment_return"]=(df["GI_Sentiment(t-1)"]-df["GI_Sentiment(t-2)"])/df["GI_Sentiment(t-2)"]
df_sentiment["GI_Sentiment_ln_return"]=np.log(df_sentiment["GI_Sentiment"]/df_sentiment["GI_Sentiment(t-1)"])
df_sentiment["GI_Sentiment_ln_return(t-1)"]=np.log(df_sentiment["GI_Sentiment(t-1)"]/df_sentiment["GI_Sentiment(t-2)"])
df_sentiment["GI_SentimentOrig_ln_return"]=np.log(df_sentiment["GI_SentimentOrig"]/df_sentiment["GI_SentimentOrig(t-1)"])
df_sentiment["GI_SentimentOrig_ln_return(t-1)"]=np.log(df_sentiment["GI_SentimentOrig(t-1)"]/df_sentiment["GI_SentimentOrig(t-2)"])

df_sentiment["GI_Sentiment_ewm10"]=df_sentiment["GI_Sentiment"].ewm(10).mean()
df_sentiment["GI_Sentiment_ewm20"]=df_sentiment["GI_Sentiment"].ewm(20).mean()

df_sentiment["GI_Positive_ln_return"]=np.log(df_sentiment["GI_Positive"]/df_sentiment["GI_Positive(t-1)"])
df_sentiment["GI_Positive_ln_return(t-1)"]=np.log(df_sentiment["GI_Positive(t-1)"]/df_sentiment["GI_Positive(t-2)"])

df_sentiment["GI_Negative_ln_return"]=np.log(df_sentiment["GI_Negative"]/df_sentiment["GI_Negative(t-1)"])
df_sentiment["GI_Negative_ln_return(t-1)"]=np.log(df_sentiment["GI_Negative(t-1)"]/df_sentiment["GI_Negative(t-2)"])

df_sentiment["GI_Polarity(t-1)"] = df_sentiment["GI_Polarity"].shift(periods=1)
df_sentiment["GI_Subjectivity(t-1)"] = df_sentiment["GI_Subjectivity"].shift(periods=1)

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df_sentiment['year'] = [d.year for d in df_sentiment.Date]
df_sentiment['month'] = [d.strftime('%b') for d in df_sentiment.Date]
df_sentiment['day'] = [d.day_name() for d in df_sentiment.Date]
years = df_sentiment['year'].unique()
df_sentiment['month-year'] = [d.strftime('%b') + str(d.year) for d in df_sentiment.Date]
df_sentiment['quarter'] = pd.PeriodIndex(df_sentiment.Date, freq='Q')

In [ ]:
df_sentiment.rename(columns={
                "Sentiment_ln_return":"Sentiment Natural Log",
                "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive Natural Log",
                "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative Natural Log", 
                "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral Natural Log",
                "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "NewsCount_ln_return":"NewsCount Natural Log",
                "NewsCount_ln_return(t-1)":"NewsCount Natural Log(t-1)",
                "WordCount_ln_return":"WordCount Natural Log",
                "WordCount_ln_return(t-1)":"WordCount Natural Log(t-1)",
                # "MarketCap_ln_return":"MarketCap Natural Log",
                # "MarketCap_ln_return(t-1)":"MarketCap Natural Log(t-1)",
                # "MarketCapFactor_ln_return":"MarketCapFactor Natural Log",
                # "MarketCapFactor_ln_return(t-1)":"MarketCapFactor Natural Log(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Sentiment_ln_return(t-1)":"GI_Sentiment Natural Log(t-1)",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Negative_ln_return(t-1)":"GI_Negative Natural Log(t-1)",
                "GI_Positive_ln_return":"GI_Positive Natural Log",
                "GI_Positive_ln_return(t-1)":"GI_Positive Natural Log(t-1)"}
                ,inplace=True)

In [ ]:
df_sentiment = df_sentiment["2011-01-09":]

In [ ]:
df_sentiment

In [ ]:
fig, axes = plt.subplots( 4,2, figsize=(20,30), dpi= 80)
i=0
j=0
axes[i,j] = sns.barplot(x='Day of Week', y='Sentiment', data=df_sentiment, ax=axes[i,j]);i=0;j=1
axes[i,j] = sns.barplot(x='Day of Week', y='Sentiment Natural Log', data=df_sentiment, ax=axes[i,j]);i=1;j=0
axes[i,j] = sns.barplot(x='Day of Week', y='Positive', data=df_sentiment, ax=axes[i,j]);i=1;j=1
axes[i,j] = sns.barplot(x='Day of Week', y='Positive Natural Log', data=df_sentiment, ax=axes[i,j]);i=2;j=0
axes[i,j] = sns.barplot(x='Day of Week', y='Negative', data=df_sentiment, ax=axes[i,j]);i=2;j=1
axes[i,j] = sns.barplot(x='Day of Week', y='Negative Natural Log', data=df_sentiment, ax=axes[i,j]);i=3;j=0
axes[i,j] = sns.barplot(x='Day of Week', y='Neutral', data=df_sentiment, ax=axes[i,j]);i=3;j=1
axes[i,j] = sns.barplot(x='Day of Week', y='Neutral Natural Log', data=df_sentiment, ax=axes[i,j]);i=0


plt.tight_layout()
plt.suptitle("New York Times\nBarPlot 2011-01-04 to 2022-06-01",fontsize=24,weight='bold')
plt.subplots_adjust(top=0.93)
plt.savefig("/Users/Farid Srouji/Documents/Research/master/NYTimes/images/Bar/7Days.png", bbox_inches='tight')
plt.show()